# **Simulating an E-commerce Marketplace Database with Python and SQLite**

This project explores how to design and populate a relational database that simulates an e-commerce marketplace.

The goal was to create a structured dataset that resembles real marketplace activity rather than purely random data.

Using Python, SQLite, and the Faker library, I generated synthetic data for users, products, categories, orders, and reviews while introducing constraints to make the dataset more realistic (e.g., aligned usernames and emails, product-brand relationships, and probabilistic order generation).

**Tools used:**
Python, SQLite, Pandas, NumPy, Faker, SQLite DB Browser

# **Install and Import Essential Libraries**

Here we would be importing and installing the required Python Libraries needed for our database creation.


1.   Numpy, Pandas - for numerical computation and data handling
2.   Faker - to generate random simulated data
1.   Sqlite3 - to interact with database

In [67]:
!pip install faker

import numpy as np
import pandas as pd
import sqlite3, random
from faker import Faker

fake = Faker()

# Seeds are set such that random outputs are conisitent throughout our work
np.random.seed(42)
random.seed(42)
Faker.seed(42)

### **Importing Database and Tables therein**

In the subsequent cell, we would be importing my SQLite database *"my_marketplace_simulation"* and retrieving the names of all Tables therein, storing the result in a Pandas DataFrame.

*[my_marketplace_simulation](https://drive.google.com/file/d/1vEJMVAuclVvw8DxvSMSEFylY4Tzz1B_U/view?usp=sharing)*


In [68]:
mp_db_path = "/content/my_marketplace_simulationdb.db"

# Building a connection to database & enabling foreign key support
conn = sqlite3.connect(mp_db_path)
conn.execute("PRAGMA foreign_keys = ON;")

# Retriving all created table names
tables = pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table' AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""", conn)

# Display
tables

,name
0,categories
1,order_items
2,orders
3,products
4,reviews
5,users


In [69]:
# List of tables that should be cleared
tables = [
    "reviews", "order_items", "orders", "products", "users", "categories"
    ]

# Retrieve the names of all existing tables in the connected database
existing = set(pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table' AND name NOT LIKE 'sqlite_%';
""", conn)["name"])

# Use a transaction to safely delete records from tables that exist
with conn:
    for t in tables:
        # Check if the table exists before attempting to delete data
        if t in existing:
            conn.execute(f"DELETE FROM {t};")

# Print confirmation message after clearing the tables
print("Cleared data (existing tables only).")

Cleared data (existing tables only).


### **Creating and Storing Categories**

Here we list and convert a our set of categories to Pandas DataFrame and insert it into our database.


In [70]:
# Itemize a list of product that would be our categories in the Database
categories = [
    "Appliances",
    "Home & Garden",
    "Furniture & Lights",
    "Sports & Travel",
    "Fashion",
    "Books",
    "Health & Beauty",
    "Toys & Games",
    "Automotive",
    "Groceries",
    "Office Supplies",
    "Gift and Jewellery"
]

# Convert categories into Pandas DataFrame
categories_df = pd.DataFrame({"category_name": categories})

# Insert DataFrame into the categories table
categories_df.to_sql("categories", conn, if_exists="append", index=False)

print("Inserted categories:", len(categories_df))

Inserted categories: 12


### **Generate Email Addresses for "users" Table**

We create functions that generates user's email addresses based on their first & last names. This is done by randomizing the prefix between user's names and numbers.


In [71]:
# Create a list of common email domains
EMAIL_DOMAINS = np.array([
    "gmail.com", "yahoo.com", "hotmail.com", "outlook.com",
    "icloud.com", "proton.me", "aol.com", "live.com"
])

# Clean text string for use in email adress
def clean_token(s: str) -> str:
    # lowercase, remove non-letters/numbers, collapse spaces
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

# Produce random email address which combines user's first and/or last names
def make_email(first: str, last: str) -> str:
    f = clean_token(first)
    l = clean_token(last)

    patterns = [
        lambda f,l: f,
        lambda f,l: l,
        lambda f,l: f + l,
        lambda f,l: l + f,
        lambda f,l: f + "." + l,
        lambda f,l: f + "_" + l,
        lambda f,l: f[0] + l,
        lambda f,l: f + l[0],
        lambda f,l: l + f[0],
        lambda f,l: f[0] + "." + l,
    ]

    base = random.choice(patterns)(f, l)

    # randomly add digits to prefix
    roll = random.random()
    if roll < 0.55:
        suffix = ""
    elif roll < 0.85:
        suffix = str(random.randint(0, 9))
    elif roll < 0.97:
        suffix = str(random.randint(10, 99))
    else:
        suffix = str(random.randint(100, 999))

    domain = str(np.random.choice(EMAIL_DOMAINS))
    return f"{base}{suffix}@{domain}"

# Generate a list of unique emails
def generate_unique_emails(first_names, last_names):
    used = set()
    emails = []
    for fn, ln in zip(first_names, last_names):
        # try until unique
        for _ in range(50):
            e = make_email(fn, ln)
            if e not in used:
                used.add(e)
                emails.append(e)
                break
        else:
            e = f"{clean_token(fn)}{clean_token(ln)}\
            {random.randint(1000,9999)}@{str(np.random.choice(EMAIL_DOMAINS))}"
            emails.append(e)

    return emails

### **Generating User Data**

Here we create 800 entries of user data for our database by using the Faker Library and Numpy Random Sampling. For names we ensured they matched corresponding gender value to a high degree.


In [72]:
n_users = 800

# Gender setup and probability of occurence
genders = np.array(['Female','Male','Non-binary','Prefer not to say'])
gender_probabilities = np.array([0.56, 0.39, 0.03, 0.02])

# List of user's country
country_list = np.array([
    "United Kingdom", "United States", "Canada", "Australia", "Ireland",
    "Germany", "France", "Netherlands", "Spain", "Italy",
    "Sweden", "Norway", "Denmark", "Switzerland", "Belgium",
    "Poland", "Portugal", "Greece", "Turkey", "United Arab Emirates",
    "India", "Pakistan", "Nigeria", "South Africa", "Kenya"
])

# Use Zipf-like distribution to generate country probability
alpha = 1.8
ranks = np.arange(1, len(country_list) + 1)
weights = 1 / (ranks ** alpha)
country_probs = weights / weights.sum()

# Create registration dates
reg_dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=365*5)

# create gender distribution
gender_sample = np.random.choice(genders, size=n_users, p=gender_probabilities)

# Link names creation to gender value
first_names = []
last_names = [fake.last_name() for _ in range(n_users)]

for g in gender_sample:
    if g == "Male":
        first_names.append(fake.first_name_male())
    elif g == "Female":
        first_names.append(fake.first_name_female())
    else:
        # Non-binary / Prefer not to say
        first_names.append(fake.first_name())

# Create users DataFrame
users_df = pd.DataFrame({
    "first_name": first_names,
    "last_name": last_names,
    "email": generate_unique_emails(first_names, last_names),
    "age": np.random.randint(16, 81, size=n_users),
    "gender": gender_sample,
    "country": np.random.choice(country_list, size=n_users, p=country_probs),
    "registration_date": pd.to_datetime(
        np.random.choice(reg_dates, size=n_users)).strftime("%Y-%m-%d")
})

# Insert dataset into Database (users table)
users_df.to_sql("users", conn, if_exists="append", index=False)
print("Inserted users:", len(users_df))

# Display
users_df.head()

Inserted users: 800


,first_name,last_name,email,age,gender,country,registration_date
0,Diane,Fowler,fowler@live.com,22,Female,Sweden,2025-11-02
1,Kenneth,Johnson,kenneth.johnson@aol.com,44,Non-binary,United Kingdom,2023-10-13
2,Kevin,Hill,kevinhill8@yahoo.com,26,Male,United States,2022-05-07
3,Zachary,Walker,walker0@gmail.com,48,Male,Portugal,2026-01-06
4,Katherine,Doyle,katherine@outlook.com,21,Female,United Kingdom,2021-03-26


### **Defining Product Catalog, Category Mapping, Validation Checks and Allowed Product Conditions**

In subsequent sessions we define the product catelog using;

*   condition-based pricing
*   jitter - for price flunctuations

*   name to id mapping for category
*   product dictionary


*   categroy conditions per product for realism

In [73]:
# Condition pricing hierarchy
CONDITION_MULTIPLIER = {
    "New": 1.00,
    "Renewed": 0.82,
    "Used": 0.68
}

# Using jitter to create small ramdom price variation
def jitter(x, pct=0.08):
    return round(float(x) * random.uniform(1 - pct, 1 + pct), 2)

# check categories in Database
cat_map_df = pd.read_sql_query("SELECT category_id,\
category_name FROM categories;", conn)
cat_map = dict(zip(cat_map_df["category_name"], cat_map_df["category_id"]))

# The full product catalog (dictionary)

PRODUCT_CATALOG = {

    "Apple": [
        ("iPhone 15 Pro", "Appliances", 999),
        ("MacBook Air M2", "Appliances", 1099),
        ("MacBook Pro 14", "Appliances", 1699),
        ("iPad Air", "Appliances", 599),
        ("AirPods Pro", "Appliances", 249),
        ("Apple Watch Series", "Appliances", 399),
    ],

    "Samsung": [
        ("Galaxy S24", "Appliances", 799),
        ("Galaxy S24 Ultra", "Appliances", 1299),
        ("Galaxy Tab S9", "Appliances", 699),
        ("Samsung 55in Smart TV", "Appliances", 699),
        ("Galaxy Buds", "Appliances", 149),
    ],

    "Sony": [
        ("PlayStation 5", "Toys & Games", 499),
        ("WH-1000XM5 Headphones", "Appliances", 399),
        ("Sony Bravia 55in TV", "Appliances", 799),
        ("Sony Soundbar", "Appliances", 299),
    ],

    "LG": [
        ("LG 55in OLED TV", "Appliances", 1199),
        ("LG Refrigerator", "Appliances", 999),
        ("LG Washing Machine", "Appliances", 749),
    ],

    "Panasonic": [
        ("Panasonic Microwave", "Appliances", 149),
        ("Panasonic Lumix Camera", "Appliances", 899),
    ],

    "Philips": [
        ("Philips Air Fryer", "Appliances", 179),
        ("Philips Electric Shaver", "Health & Beauty", 129),
        ("Philips Hue Starter Kit", "Home & Garden", 199),
    ],

    "Bose": [
        ("Bose QuietComfort 45", "Appliances", 329),
        ("Bose Smart Speaker", "Appliances", 249),
    ],

    "JBL": [
        ("JBL Flip 6", "Appliances", 129),
        ("JBL PartyBox", "Appliances", 499),
    ],

    "Dell": [
        ("Dell XPS 13", "Office Supplies", 1299),
        ("Dell Inspiron 15", "Office Supplies", 699),
        ("Dell UltraSharp Monitor", "Office Supplies", 449),
    ],

    "HP": [
        ("HP Pavilion 15", "Office Supplies", 749),
        ("HP Envy 13", "Office Supplies", 999),
        ("HP OfficeJet Printer", "Office Supplies", 179),
    ],

    "Lenovo": [
        ("Lenovo ThinkPad X1", "Office Supplies", 1399),
        ("Lenovo IdeaPad 5", "Office Supplies", 699),
    ],

    "Asus": [
        ("Asus ZenBook 14", "Office Supplies", 899),
        ("Asus ROG Strix Laptop", "Office Supplies", 1499),
    ],

    "Acer": [
        ("Acer Aspire 5", "Office Supplies", 649),
        ("Acer Predator Gaming Laptop", "Office Supplies", 1599),
    ],

    "Microsoft": [
        ("Surface Laptop 5", "Office Supplies", 1299),
        ("Surface Pro 9", "Office Supplies", 999),
        ("Xbox Series X", "Toys & Games", 499),
    ],

    "Nike": [
        ("Air Force 1", "Fashion", 115),
        ("Air Max 90", "Fashion", 140),
        ("Nike Tech Fleece Hoodie", "Fashion", 120),
    ],

    "Adidas": [
        ("Ultraboost 22", "Fashion", 190),
        ("Stan Smith", "Fashion", 110),
        ("Adidas Track Jacket", "Fashion", 80),
    ],

    "Puma": [
        ("Puma Suede Classic", "Fashion", 90),
        ("Puma Running Shoes", "Fashion", 120),
        ("Puma Training Hoodie", "Fashion", 85),
    ],

     "Penguin": [
        ("Atomic Habits (Paperback)", "Books", 12),
        ("Deep Work (Paperback)", "Books", 14),
        ("The Psychology of Money (Paperback)", "Books", 13),
    ],

    "Heinz": [
        ("Tomato Ketchup 1kg", "Groceries", 4),
        ("Baked Beans 415g", "Groceries", 2),
    ],

    "Kellogg's": [
        ("Corn Flakes 500g", "Groceries", 3),
        ("Special K 500g", "Groceries", 4),
    ],

    "Castrol": [
        ("Engine Oil 5W-30 5L", "Automotive", 39),
        ("Brake Fluid 1L", "Automotive", 12),
    ],

    "Michelin": [
        ("All-Season Tyre", "Automotive", 95),
        ("Performance Tyre", "Automotive", 145),
    ],

    "Samsonite": [
        ("Carry-On Suitcase", "Sports & Travel", 149),
        ("Travel Backpack", "Sports & Travel", 89),
    ],

    "Pandora": [
        ("Charm Bracelet", "Gift and Jewellery", 59),
        ("Silver Necklace", "Gift and Jewellery", 79),
    ],

    "Swarovski": [
        ("Crystal Earrings", "Gift and Jewellery", 89),
        ("Crystal Pendant", "Gift and Jewellery", 119),
    ],
}

# Authenticate catalog categories
missing_cats = sorted(
    {c for brand in PRODUCT_CATALOG for _, c, _ in PRODUCT_CATALOG[brand]}\
    - set(cat_map.keys()))
if missing_cats:
    raise ValueError(
        "PRODUCT_CATALOG references categories not in DB: "
        + ", ".join(missing_cats)
    )

In [74]:
# Describe product conditions for each category
CATEGORY_CONDITIONS = {
    # Electronics & devices: all conditions
    "Appliances": ["New", "Renewed", "Used"],
    "Office Supplies": ["New", "Renewed", "Used"],
    "Toys & Games": ["New", "Renewed", "Used"],

    # Furniture / home items
    "Home & Garden": ["New", "Used"],
    "Furniture & Lights": ["New", "Used"],

    # Fashion: mostly new, sometimes used
    "Fashion": ["New", "Used"],

    # Books: new or used
    "Books": ["New", "Used"],

    # Jewellery: new or used
    "Gift and Jewellery": ["New", "Used"],

    # Automotive consumables + groceries: new only
    "Automotive": ["New"],
    "Groceries": ["New"],
    "Health & Beauty": ["New"],

    # Travel goods: new or used
    "Sports & Travel": ["New", "Used"],
}

In [75]:
n_products = 250

# generate possible production date
created_dates = pd.date_range(
    end=pd.Timestamp.today().normalize(), periods=365*2)

# Flatten product catalog into a list for sampling
catalog_rows = []
for brand, items in PRODUCT_CATALOG.items():
  for model_name, category_name, base_price in items:
    catalog_rows.append((brand, model_name, category_name, base_price))

# Create DataFrame for the Full product catalog
catalog_df = pd.DataFrame(
    catalog_rows,
    columns=["brand", "model_name", "category_name", "base_price_new"]
)

# Sample products from catalog (with replacement, so popular models can repeat)
sample_idx = np.random.choice(catalog_df.index, size=n_products, replace=True)
sample = catalog_df.loc[sample_idx].reset_index(drop=True)

# --- NEW: Choose condition PER ROW based on category rules ---
chosen_conditions = []
for cat in sample["category_name"]:
  allowed = CATEGORY_CONDITIONS.get(cat, ["New"]) # fallback
  chosen_conditions.append(random.choice(allowed))
chosen_conditions = np.array(chosen_conditions)

# --- Pricing: still works exactly as before ---
prices = []
for base_new, cond in zip(sample["base_price_new"], chosen_conditions):
  base = float(base_new) * CONDITION_MULTIPLIER[cond]
  prices.append(jitter(base, pct=0.08))

products_df = pd.DataFrame({
    "category_id": sample["category_name"].map(cat_map).astype(int),
    "product_name": sample["model_name"],
    "brand": sample["brand"],
    "price": prices,
    "stock_quantity": np.random.randint(0, 600, size=n_products),
    "condition_level": chosen_conditions,
    "created_date": pd.to_datetime(
        np.random.choice(created_dates, size=n_products)).strftime("%Y-%m-%d")
})

# Optional: avoid identical duplicates per (brand, product, condition)
dup_mask = products_df.duplicated(
    subset=["brand", "product_name", "condition_level"], keep=False)

products_df.to_sql("products", conn, if_exists="append", index=False)

print("Inserted products:", len(products_df))

# Display
products_df.head()

Inserted products: 250


,category_id,product_name,brand,price,stock_quantity,condition_level,created_date
0,1,LG Refrigerator,LG,780.57,561,Renewed,2024-03-11
1,12,Silver Necklace,Pandora,54.43,112,Used,2024-04-23
2,1,Apple Watch Series,Apple,253.56,169,Used,2024-06-16
3,1,LG Washing Machine,LG,510.17,467,Used,2026-02-28
4,7,Philips Electric Shaver,Philips,135.37,13,New,2024-11-30


### **Generating Order Items for Each Order**

This section creates a dataset of 1500 orders placed by users. We have linked each order to random users generated earlier and stored in our database.

NOTE: the **total_amount** column is set to 0 as its actual value would be calculated in subsequent sections of this workbook.

In [76]:
n_orders = 1500

# fetch users and registration date from database
users_df = pd.read_sql_query("SELECT user_id, registration_date FROM users;",\
                             conn)

# order status and probability of occurence
order_statuses = ["Processing", "Shipped", "Delivered", "Cancelled"]
status_probs = [0.15, 0.25, 0.50, 0.10]

orders = []

# geenerate random orders
for _ in range(n_orders):

    user = users_df.sample(1).iloc[0]

    order_date = pd.to_datetime(
        np.random.choice(created_dates)
    )

    orders.append({
        "user_id": int(user["user_id"]),
        "order_date": order_date.strftime("%Y-%m-%d"),
        "order_status": np.random.choice(order_statuses, p=status_probs),
        "total_amount": 0.0
    })

# convert order to database
orders_df = pd.DataFrame(orders)

# insert order into database table "orders"
orders_df.to_sql("orders", conn, if_exists="append", index=False)

print("Inserted orders:", len(orders_df))

# Display
orders_df.head()

Inserted orders: 1500


,user_id,order_date,order_status,total_amount
0,49,2026-02-07,Shipped,0.0
1,529,2026-02-01,Delivered,0.0
2,794,2024-12-06,Delivered,0.0
3,700,2024-12-15,Shipped,0.0
4,201,2024-08-19,Cancelled,0.0


In [77]:
# reset order items and totals
with conn:
    conn.execute("DELETE FROM order_items;")
    conn.execute("UPDATE orders SET total_amount = 0;")
print("Cleared order_items and reset totals.")

Cleared order_items and reset totals.


In [78]:
# Pull ids from DB (robust)
order_ids = pd.read_sql_query("SELECT order_id FROM orders ORDER BY order_id;",
                              conn)["order_id"].to_numpy()
product_rows = pd.read_sql_query("SELECT product_id, price FROM products;",
                                 conn)
product_ids = product_rows["product_id"].to_numpy()

price_map = dict(zip(product_rows["product_id"], product_rows["price"]))

# Items per order distribution (realistic skew)
items_per_order_choices = np.array([1, 2, 3, 4, 5])
items_per_order_probs   = np.array([0.45, 0.30, 0.15, 0.07, 0.03])

# Quantity distribution: mostly 1–3, sometimes larger up to 50
qty_choices = np.array([1,2,3,4,5,6,7,8,9,10,20,30,40,50])
qty_probs   = np.array([0.28,0.22,0.16,0.08,0.06,0.05,0.03,0.03,0.02,0.02,
                        0.015,0.01,0.005,0.005])
qty_probs   = qty_probs / qty_probs.sum()

rows = []

for oid in order_ids:
    k = np.random.choice(items_per_order_choices, p=items_per_order_probs)
    chosen = np.random.choice(product_ids, size=k, replace=False)

    for pid in chosen:
        qty = int(np.random.choice(qty_choices, p=qty_probs))
        unit_price = float(price_map[int(pid)])
        rows.append((int(oid), int(pid), qty, unit_price))

order_items_df = pd.DataFrame(rows, columns=["order_id","product_id",
                                             "quantity","unit_price"])

order_items_df.to_sql("order_items", conn, if_exists="append", index=False)

print("Inserted order_items:", len(order_items_df))

# Display
order_items_df.head()

Inserted order_items: 2865


,order_id,product_id,quantity,unit_price
0,1,2,3,54.43
1,1,128,6,282.68
2,1,127,4,544.58
3,1,53,2,139.14
4,2,128,7,282.68


In [79]:
with conn:
    conn.execute("""
    UPDATE orders
    SET total_amount = (
        SELECT ROUND(SUM(quantity * unit_price), 2)
        FROM order_items
        WHERE order_items.order_id = orders.order_id
    );
    """)

print("Updated total_amount for all orders.")

Updated total_amount for all orders.


In [80]:
# Foreign key check should be empty
fk_issues = pd.read_sql_query("PRAGMA foreign_key_check;", conn)
fk_issues

,table,rowid,parent,fkid


In [81]:
# Confirm totals are now non-zero for most orders
pd.read_sql_query("""
SELECT order_id, user_id, order_status, total_amount
FROM orders
ORDER BY order_id
LIMIT 5;
""", conn)

,order_id,user_id,order_status,total_amount
0,1,49,Shipped,4315.97
1,2,529,Delivered,1978.76
2,3,794,Delivered,857.37
3,4,700,Shipped,54997.45
4,5,201,Cancelled,352.12


### **Generating Product Reviews from Delivered Purchases**

Here we generate user reviews for our review dataset based on delivered orders. The Key characteristics of this review includes:

*   the use of user_id and product_id from delivered orders
*   removal to duplicates to ensure each user review once

*   rating ranging from 1 to 5 and following a skewed distribution favouring high ratings
*   review content, picked from a set of positive, mild and negative comments tied to its predominant rating value

*   missen review content (=null), which examplifies actual user pattern of rating without review content

In [82]:
delivered_purchases = pd.read_sql_query("""
SELECT
    o.user_id,
    oi.product_id,
    o.order_date
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'Delivered'
""", conn)

# Remove duplicates to respect UNIQUE(user_id, product_id)
delivered_purchases = delivered_purchases.drop_duplicates(
    subset=["user_id","product_id"]
).reset_index(drop=True)

# Number of reviews to generate
target_reviews = 1200
n_reviews = min(target_reviews, len(delivered_purchases))

sample_reviews = delivered_purchases.sample\
 (n_reviews, random_state=42).reset_index(drop=True)

# Rating distribution (realistic skew)
ratings = np.array([1,2,3,4,5])
rating_probs = np.array([0.05,0.08,0.17,0.35,0.35])

ratings_sample = np.random.choice(ratings, size=n_reviews, p=rating_probs)

# Review text pools
positive_reviews = [
    "Excellent quality and fast delivery.",
    "Very satisfied with this product.",
    "Works exactly as expected.",
    "Great value for the price.",
    "Highly recommend this item.",
]

neutral_reviews = [
    "The product is okay for the price.",
    "Average quality but works fine.",
    "Nothing special but does the job.",
    "It meets basic expectations.",
]

negative_reviews = [
    "The product did not meet expectations.",
    "Quality could be much better.",
    "Not satisfied with this purchase.",
    "Would not recommend this item.",
]

def generate_review_text(rating):

    # allow some missing reviews
    if random.random() < 0.12:
        return None

    if rating >= 4:
        return random.choice(positive_reviews)

    elif rating == 3:
        return random.choice(neutral_reviews)

    else:
        return random.choice(negative_reviews)

review_contents = []
review_dates = []

for rating, od in zip(ratings_sample, sample_reviews["order_date"]):

    order_dt = pd.to_datetime(od)

    # review occurs after order date
    rd = order_dt + pd.to_timedelta(random.randint(1,30), unit="D")

    review_dates.append(rd.strftime("%Y-%m-%d"))
    review_contents.append(generate_review_text(rating))

reviews_df = pd.DataFrame({
    "user_id": sample_reviews["user_id"].astype(int),
    "product_id": sample_reviews["product_id"].astype(int),
    "rating": ratings_sample,
    "review_content": review_contents,
    "review_date": review_dates
})

reviews_df.to_sql("reviews", conn, if_exists="append", index=False)

print("Inserted reviews:", len(reviews_df))

# Display
reviews_df.head()

Inserted reviews: 1200


,user_id,product_id,rating,review_content,review_date
0,390,228,2,The product did not meet expectations.,2026-02-14
1,339,90,4,Highly recommend this item.,2024-09-03
2,294,233,5,Very satisfied with this product.,2024-10-27
3,23,129,4,Excellent quality and fast delivery.,2025-08-09
4,209,245,4,Great value for the price.,2025-10-02


In [83]:
# Download Database File
from google.colab import files
files.download('/content/my_marketplace_simulationdb.db')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>